# Survival Analysis of Underpricing in Indian SME IPOs (NSE Emerge 2013–2024)
**Author:** Rajat Sambare (2022A2PS1727H) | **Supervised by:** Prof. Sravani Bharandev (BITS Pilani)

This reproducible notebook executes the end-to-end econometric and machine learning survival analysis pipeline on 436 underpriced SME IPOs.
- **Kaplan-Meier Estimation & Stratified Log-Rank Tests**
- **L1-Penalized Cox Proportional Hazards Regression**
- **Random Survival Forest with 5-Fold Cross-Validation**

In [ ]:
# Install required dependencies
!pip install --quiet lifelines scikit-survival tabulate

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sklearn.model_selection import KFold

sns.set_theme(style='whitegrid')
print('Packages loaded successfully.')

## 1. Load Compiled SME Survival Dataset (N=436)

In [ ]:
# Load survival dataset directly from repository or upload
data_url = 'https://raw.githubusercontent.com/Ry-tZ/sme-ipo-survival-analysis/master/data/processed/sme_survival_data.csv'
try:
    df = pd.read_csv(data_url)
    print('Loaded dataset from GitHub repository.')
except Exception:
    df = pd.read_csv('sme_survival_data.csv')
    print('Loaded dataset from local path.')

df['log_traded_qty'] = np.log1p(df['traded_qty_l1'].fillna(0))
df['is_hot_period'] = (df['listing_year'] >= 2023).astype(int)
df['pe_ratio_clipped'] = np.clip(df['pe_ratio'].fillna(15.0), -100, 300)
df['debt_to_asset_ratio'] = df['debt_to_asset_ratio'].fillna(0.45)

print(f'Total cohort: {len(df)} firms')
print(f'Events observed (Price <= Issue Price): {df["event"].sum()} ({df["event"].mean()*100:.1f}%)')
print(f'Right-censored: {(1 - df["event"]).sum()} ({(1 - df["event"]).mean()*100:.1f}%)')
df.head()

## 2. Non-Parametric Kaplan-Meier Survival Analysis

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(df['time'], event_observed=df['event'], label='All SME IPOs (N=436)')

plt.figure(figsize=(10, 5))
kmf.plot_survival_function(ci_show=True, color='#0b57d0')
plt.title('Kaplan-Meier Survival Curve: SME IPO Underpricing Duration', fontsize=13, fontweight='bold')
plt.xlabel('Trading Days After Listing')
plt.ylabel('Probability of Remaining Underpriced S(t)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

print(f'Median Underpricing Duration: {kmf.median_survival_time_:.1f} trading days')

In [ ]:
# Stratified Comparison: Hot Era (2023-2024) vs Pre-2023 (2013-2022)
hot_mask = df['is_hot_period'] == 1
kmf_hot = KaplanMeierFitter()
kmf_cold = KaplanMeierFitter()

kmf_hot.fit(df.loc[hot_mask, 'time'], df.loc[hot_mask, 'event'], label='Hot Era (2023-2024, N=257)')
kmf_cold.fit(df.loc[~hot_mask, 'time'], df.loc[~hot_mask, 'event'], label='Pre-2023 (2013-2022, N=179)')

lr_res = logrank_test(df.loc[hot_mask, 'time'], df.loc[~hot_mask, 'time'],
                      df.loc[hot_mask, 'event'], df.loc[~hot_mask, 'event'])

plt.figure(figsize=(10, 5))
ax = kmf_hot.plot_survival_function(color='#d93025')
kmf_cold.plot_survival_function(ax=ax, color='#1e8e3e')
plt.title(f'SME Underpricing Duration: Hot Era vs Previous Cycles (Log-Rank p={lr_res.p_value:.4e})', fontsize=12, fontweight='bold')
plt.xlabel('Trading Days After Listing')
plt.ylabel('Probability of Remaining Underpriced')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 3. Semi-Parametric Cox Proportional Hazards Regression

In [ ]:
cox_cols = ['time', 'event', 'listing_gain_pct', 'log_traded_qty', 'eps', 'pe_ratio_clipped', 'debt_to_asset_ratio', 'is_hot_period']
cph = CoxPHFitter(penalizer=0.1, l1_ratio=0.5)
cph.fit(df[cox_cols].dropna(), duration_col='time', event_col='event')
cph.print_summary()

plt.figure(figsize=(9, 4.5))
cph.plot()
plt.title('Cox Proportional Hazards Model: Impact of Covariates on Hazard of Underpricing Termination', fontsize=11, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 4. Machine Learning: Random Survival Forest (RSF)

In [ ]:
features = ['listing_gain_pct', 'log_traded_qty', 'eps', 'pe_ratio_clipped', 'debt_to_asset_ratio', 'is_hot_period']
X = df[features].copy().fillna(0)
y = Surv.from_dataframe('event', 'time', df)

# 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []
for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    rsf = RandomSurvivalForest(n_estimators=100, min_samples_split=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rsf.fit(X_train, y_train)
    scores.append(rsf.score(X_test, y_test))

print(f'Random Survival Forest 5-Fold Mean C-Index: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})')

# Fit full model and plot predicted curves
final_rsf = RandomSurvivalForest(n_estimators=100, min_samples_split=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
final_rsf.fit(X, y)
surv_funcs = final_rsf.predict_survival_function(X.iloc[:4])

plt.figure(figsize=(10, 5))
for i, fn in enumerate(surv_funcs):
    plt.step(fn.x, fn(fn.x), where='post', label=f'Firm {i+1} ({df.iloc[i]["company_name"][:20]}...)')
plt.title('Random Survival Forest: Individual Predicted Survival Trajectories', fontsize=12, fontweight='bold')
plt.xlabel('Trading Days After Listing')
plt.ylabel('Estimated Survival Probability')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()